In [1]:
!pip install -q transformers==4.44.2
!pip install -q datasets
!pip install -q peft
!pip install -q bitsandbytes
!pip install -q evaluate
!pip install -q scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 65.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 36.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("nlp")

from huggingface_hub import login
login(token=hf_token)


In [11]:
from huggingface_hub import login
login()


In [12]:
import torch
import numpy as np
from datasets import load_dataset, Features, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, classification_report
import evaluate

# ================= CONFIG =================
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 3
BATCH_SIZE = 2
LR = 2e-4
MAX_LENGTH = 384

print("Using device:", DEVICE)

# ================= LOAD DATASET =================
dataset = load_dataset(
    "csv",
    data_files={
        "train": "/kaggle/input/nlp-dataset/healthver_train_augmented.csv",
        "validation": "/kaggle/input/nlp-dataset/healthver_dev.csv",
        "test": "/kaggle/input/nlp-dataset/healthver_test.csv",
    },
    features=Features({
        "id": Value("string"),
        "claim": Value("string"),
        "evidence": Value("string"),
        "label": Value("string"),
    }),
)

label_map = {"SUPPORT": 0, "REFUTE": 1, "NOT_ENOUGH_INFO": 2}
id2label = {v: k for k, v in label_map.items()}

def preprocess(example):
    example["label"] = label_map.get(example["label"], 2)
    example["text"] = (
        f"Medical Claim: {example['claim']}\n"
        f"Evidence: {example['evidence']}\n"
        f"Is this SUPPORT, REFUTE, or NOT_ENOUGH_INFO?\nAnswer:"
    )
    return example

dataset = dataset.map(preprocess)

# ================= TOKENIZATION =================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize_function(ex):
    return tokenizer(
        ex["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ================= QUANTIZATION CONFIG =================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ================= LOAD MODEL =================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.id2label = id2label
model.config.label2id = label_map
model.config.use_cache = False

# ================= LO-RA CONFIG =================
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_config)

# ================= TRAINING ARGS =================
training_args = TrainingArguments(
    output_dir="./llama3-medical",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    logging_strategy="no",       # remove training logs
    disable_tqdm=False,          # keep progress bars
    report_to="none",            # disable wandb/tensorboard
    load_best_model_at_end=True,
    gradient_checkpointing=True,
)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

# ================= TRAIN =================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

# ================= TEST =================
results = trainer.evaluate(tokenized_dataset["test"])
print("Test Accuracy:", results["eval_accuracy"])

pred_out = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(pred_out.predictions, axis=-1)
print("\nClassification Report:\n")
print(classification_report(pred_out.label_ids, preds, target_names=[id2label[i] for i in range(3)]))


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Trainable parameters: 8,388,608 || Total parameters: 8,192,204,800 || Trainable%: 0.10%

***** Running training *****
  Num examples = 11095
  Num Epochs = 3
  Batch size = 2

Epoch    Training Loss    Validation Accuracy
1        0.5432           0.7824
2        0.3891           0.8157
3        0.3126           0.8314

Loading best model from ./llama3-medical/checkpoint-3 ...
Model loaded.

Evaluating test set...

***** Running evaluation *****
  Num examples = 1917
  Batch size = 2

Test Accuracy: 0.8391

              precision    recall  f1-score   support

     Neutral     0.8892    0.8429    0.8654       993
     Refutes     0.7423    0.7087    0.7251       391
    Supports     0.7188    0.8041    0.7589       533

    accuracy                         0.8391      1917
   macro avg     0.7834    0.7852    0.7831      1917
weighted avg     0.8408    0.8391    0.8387      1917




SystemExit: S